In [1]:
!pip install tokenizers tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import bz2
import xml.etree.ElementTree as ET
import urllib.request
import os
import math
from tqdm import tqdm

In [3]:
urls = [
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwiktionary/2026-04-01/xml/bzip2/enwiktionary-2026-04-01-p7p11575490.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwikiquote/2026-04-01/xml/bzip2/enwikiquote-2026-04-01-p1p305881.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/enwikibooks/2026-04-01/xml/bzip2/enwikibooks-2026-04-01-p1p482395.xml.bz2",
    "https://dumps.wikimedia.org/other/mediawiki_content_current/simplewiki/2026-04-01/xml/bzip2/simplewiki-2026-04-01-p1p1261009.xml.bz2",                    
]

os.makedirs("data", exist_ok=True)
txt_corpus_path = "data/corpus.txt"

def download_and_extract_text(urls, output_file):
    """Downloads bz2 wiki dumps and extracts basic text to a single file."""
    print("Downloading and extracting Wikipedia text...")
    with open(output_file, "w", encoding="utf-8") as out_f:
        for url in urls:
            filename = os.path.join("data", url.split('/')[-1])
            if not os.path.exists(filename):
                print(f"Downloading {filename}...")
                urllib.request.urlretrieve(url, filename)
            
            print(f"Extracting {filename}...")
            # Stream the bz2 file to save RAM
            with bz2.open(filename, "rt", encoding="utf-8") as bz2_file:
                try:
                    context = ET.iterparse(bz2_file, events=("end",))
                    for event, elem in context:
                        # Find tags ending with 'text' (ignoring namespaces)
                        if elem.tag.endswith('text') and elem.text:
                            # Basic cleanup (in a real scenario, use mwparserfromhell)
                            text = elem.text.strip()
                            if len(text) > 50: # Skip tiny entries
                                out_f.write(text + "\n\n")
                        elem.clear() # Free memory
                except ET.ParseError:
                    print(f"Reached end of valid XML in {filename}")

download_and_extract_text(urls, txt_corpus_path)

Extracting data/enwiktionary-2026-04-01-p7p11575490.xml.bz2...
Extracting data/enwikiquote-2026-04-01-p1p305881.xml.bz2...
Extracting data/enwikibooks-2026-04-01-p1p482395.xml.bz2...
Extracting data/simplewiki-2026-04-01-p1p1261009.xml.bz2...


In [4]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"], vocab_size=32000)

# 1. Create a generator to yield text in chunks (saves RAM)
def batch_iterator(batch_size=10000):
    with open(txt_corpus_path, "r", encoding="utf-8") as f:
        batch = []
        for line in f:
            line = line.strip()
            if line:
                batch.append(line)
            if len(batch) == batch_size:
                yield batch
                batch = []
        if batch:
            yield batch

print("Training Tokenizer via streaming...")
# 2. Train using the iterator instead of the file path
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

tokenizer.save("tokenizer.json")
print("Tokenizer saved!")

Training Tokenizer via streaming...



Tokenizer saved!


In [5]:
import numpy as np

# 1. Initialize an empty list to hold the token IDs
token_ids = []

print("Starting full dataset tokenization...")
# 2. Use a loop to encode the text without crashing RAM
with open(txt_corpus_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            # Encode the line and add the IDs to our list
            encoded = tokenizer.encode(line)
            token_ids.extend(encoded.ids)

# 3. Convert to a high-performance NumPy array
# We use uint32 to save space (since your vocab is 32,000, 
# you don't need 64-bit integers)
token_array = np.array(token_ids, dtype=np.uint32)

# 4. Save to a binary file
np.save("corpus_tokens.npy", token_array)

print(f"Success! Processed {len(token_array)} total tokens.")
print("Files to download for your next notebook: 'tokenizer.json' and 'corpus_tokens.npy'")

Starting full dataset tokenization...
